### Modelling

#### Importing Packages and Loading Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [3]:
project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"
models_dir = project_root / "models"
figures_dir = project_root / "reports" / "figures"

In [4]:
df = pd.read_parquet(processed_dir / "model_features.parquet")
df["month_date"] = pd.to_datetime(df["month_date"])

#### Features and Train/Test

In [5]:
# Encoding and Splitting
df["ward_id"] = df["ward_code"].astype("category").cat.codes
df["type_id"] = df["crime_type"].astype("category").cat.codes

feature_cols = [
    "lag_1", "lag_2", "lag_3", "lag_12", "roll_mean_3",
    "month_num",
    "ward_id", "type_id",
]
target_col = "crime_count"

In [6]:
split_date = "2025-09-01"
train_mask = df["month_date"] <  split_date
test_mask  = df["month_date"] >= split_date

In [7]:
X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, target_col]
X_test,  y_test  = df.loc[test_mask,  feature_cols], df.loc[test_mask,  target_col]

In [10]:
print(f"Train: {len(X_train):,} rows  ({df.loc[train_mask, 'month'].min()} to {df.loc[train_mask, 'month'].max()})")
print(f"Test:  {len(X_test):,} rows  ({df.loc[test_mask,  'month'].min()} to {df.loc[test_mask,  'month'].max()})")

Train: 133,570 rows  (2024-02 to 2025-08)
Test:  28,120 rows  (2025-09 to 2025-12)


#### Naive Baseline

In [13]:
naive_pred = df.loc[test_mask, "lag_1"].values

mae_naive  = mean_absolute_error(y_test, naive_pred)
rmse_naive = np.sqrt(mean_squared_error(y_test, naive_pred))

print(f"  MAE  = {mae_naive}")
print(f"  RMSE = {rmse_naive}")

  MAE  = 4.343705547652916
  RMSE = 7.813829972143648


#### Random Forest

In [19]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred_rf  = rf.predict(X_test)
mae_rf   = mean_absolute_error(y_test, pred_rf)
rmse_rf  = np.sqrt(mean_squared_error(y_test, pred_rf))

print(f"  MAE  = {mae_rf}")
print(f"  RMSE = {rmse_rf}")

  MAE  = 3.733963632516558
  RMSE = 7.201020836737165


#### XGBoost

In [25]:
xgb = XGBRegressor(
    objective="count:poisson",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    n_jobs=-1,
    random_state=42,
)
xgb.fit(X_train, y_train)

pred_xgb  = xgb.predict(X_test)
mae_xgb   = mean_absolute_error(y_test, pred_xgb)
rmse_xgb  = np.sqrt(mean_squared_error(y_test, pred_xgb))

print(f"  MAE  = {mae_xgb}")
print(f"  RMSE = {rmse_xgb}")

  MAE  = 3.6709237098693848
  RMSE = 8.385742585539905


#### Final Model

In [28]:
final_model = rf
final_name  = "Random Forest"
final_pred  = pred_rf
final_mae, final_rmse = mae_rf, rmse_rf

print(f"Final model: {final_name}")
print(f"  MAE  = {final_mae}")
print(f"  RMSE = {final_rmse}")

Final model: Random Forest
  MAE  = 3.733963632516558
  RMSE = 7.201020836737165


In [31]:
joblib.dump(final_model, models_dir / "best_model.joblib")

['C:\\Users\\Abdul Qudus\\Documents\\Data Portfolio\\london-crime-hotspot-prediction\\models\\best_model.joblib']

#### Summary

The three models were test on a split trained on Feb 2024 to Aug 2025, held out Sep–Dec 2025 as a real future the model never saw during training or selection.

The naive baseline just predict that next month's crime count equals last month's  gave a MAE of 4.34.

Random Forest came in at MAE 3.73, RMSE 7.20 an improvement compared to naive. XGBoost came in
slightly lower on MAE 3.67, but its RMSE was 8.39 actually worse than the naive baseline.

MAE and RMSE measure different things. MAE is the average miss and RMSE
penalises big misses much more heavily. When the two metrics disagree, the
honest answer is that the model with better RMSE is the one making more
reliable predictions.

Random Forest is our final model. For a hotspot forecaster a single very wrong prediction is worse than several mildly wrong ones. Random Forest's reliability advantage outweighs XGBoost's tiny edge on MAE. Picking on MAE
alone was not the best decision.

The model uses lag features (1, 2, 3, and 12 months back), a 3-month rolling
mean, month number, and integer-encoded ward and crime type.